In [1]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

# Model eval

## Load data

In [2]:
# load csv file of features as a pandas dataframe, when it comes to noninterpretable features 
# i think we can save them as .npy files and they can have a database 
exp_features_df=pd.read_csv("interpretable_features_dummy_for_testing.csv")

In [3]:

def get_models(random_state=42):
    return {
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=random_state
            ))
        ]),

        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),

        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                random_state=random_state
            ))
        ]),

        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=random_state
            ))
        ]),
    }
def generate_cv_predictions(features_df, feature_columns, models, target_column="label",
                            sample_id_column="cha_path", feature_set_name="default",
                            n_splits=5, random_state=42):
    X = features_df[feature_columns].copy()
    y = features_df[target_column].copy().values
    sample_ids = features_df[sample_id_column].values

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    prediction_rows = []

    for model_name, model in models.items():
        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]

            clf = clone(model)
            clf.fit(X_train, y_train)

            y_pred = clf.predict(X_test)

            row_dict = {
                "sample_id": sample_ids[test_idx],
                "model": model_name,
                "feature_set": feature_set_name,
                "fold": fold_idx,
                "y_true": y_test,
                "y_pred": y_pred,
            }

            # probabilities if available
            if hasattr(clf, "predict_proba"):
                y_proba = clf.predict_proba(X_test)
                for class_idx in range(y_proba.shape[1]):
                    row_dict[f"prob_class_{class_idx}"] = y_proba[:, class_idx]

            fold_df = pd.DataFrame(row_dict)
            prediction_rows.append(fold_df)

    predictions_df = pd.concat(prediction_rows, ignore_index=True)
    return predictions_df

def compute_metrics_from_predictions(predictions_df):
    metrics_rows = []
    per_class_rows = []

    group_cols = ["model", "feature_set"]

    for (model_name, feature_set), df_group in predictions_df.groupby(group_cols):
        y_true = df_group["y_true"].values
        y_pred = df_group["y_pred"].values
        classes = np.sort(np.unique(y_true))

        row = {
            "model": model_name,
            "feature_set": feature_set,
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        }

        prob_cols = [c for c in df_group.columns if c.startswith("prob_class_")]

        if len(prob_cols) > 0:
            y_proba = df_group[prob_cols].values

            if len(classes) == 2 and y_proba.shape[1] >= 2:
                row["roc_auc"] = roc_auc_score(y_true, y_proba[:, 1])
            elif len(classes) > 2:
                y_bin = label_binarize(y_true, classes=classes)
                row["roc_auc_ovr_macro"] = roc_auc_score(
                    y_bin,
                    y_proba,
                    multi_class="ovr",
                    average="macro"
                )

        metrics_rows.append(row)

        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class_or_avg"})
        report_df["model"] = model_name
        report_df["feature_set"] = feature_set
        per_class_rows.append(report_df)

    metrics_df = pd.DataFrame(metrics_rows)
    per_class_metrics_df = pd.concat(per_class_rows, ignore_index=True)

    return metrics_df, per_class_metrics_df

In [4]:
from pathlib import Path

exp_features_df["sample_id"] = exp_features_df["wav_path"].apply(lambda x: Path(x).stem)

FEATURE_COLUMNS = [
    "total_speaking_time_normalized",
    "interviewer_interruptions",
    "word_rate_words_per_sec",
]

models = get_models(random_state=42)

predictions_df = generate_cv_predictions(
    features_df=exp_features_df,
    feature_columns=FEATURE_COLUMNS,
    models=models,
    target_column="label",
    sample_id_column="sample_id",
    feature_set_name="basic_interpretable",
    n_splits=5,
    random_state=42
)

metrics_df, per_class_metrics_df = compute_metrics_from_predictions(predictions_df)

print(predictions_df.head())
print(metrics_df)
print(per_class_metrics_df.head())

  sample_id model          feature_set  fold  y_true  y_pred  prob_class_0  \
0     016-0   SVM  basic_interpretable     1       1       0      0.633635   
1     157-2   SVM  basic_interpretable     1       1       0      0.717935   
2     164-2   SVM  basic_interpretable     1       1       0      0.661806   
3     216-0   SVM  basic_interpretable     1       1       0      0.638678   
4     235-2   SVM  basic_interpretable     1       1       0      0.500000   

   prob_class_1  
0      0.366365  
1      0.282065  
2      0.338194  
3      0.361322  
4      0.500000  
          model          feature_set  accuracy  balanced_accuracy  \
0           KNN  basic_interpretable  0.634146           0.633201   
1  RandomForest  basic_interpretable  0.663957           0.663360   
2           SVM  basic_interpretable  0.691057           0.689286   
3       XGBoost  basic_interpretable  0.642276           0.641931   

   precision_macro  recall_macro  f1_macro   roc_auc  
0         0.634068    

In [14]:
predictions_df[(predictions_df["model"] == "SVM")&(predictions_df["fold"] == 1)&(predictions_df["y_true"] == 1)]

,sample_id,model,feature_set,fold,y_true,y_pred,prob_class_0,prob_class_1
0,016-0,SVM,basic_interpretable,1,1,0,0.633635,0.366365
1,157-2,SVM,basic_interpretable,1,1,0,0.717935,0.282065
2,164-2,SVM,basic_interpretable,1,1,0,0.661806,0.338194
3,216-0,SVM,basic_interpretable,1,1,0,0.638678,0.361322
4,235-2,SVM,basic_interpretable,1,1,0,0.500000,0.500000
5,236-0,SVM,basic_interpretable,1,1,1,0.211699,0.788301
6,238-0,SVM,basic_interpretable,1,1,1,0.190388,0.809612
7,260-2,SVM,basic_interpretable,1,1,1,0.264113,0.735887
8,269-1,SVM,basic_interpretable,1,1,1,0.282780,0.717220
9,270-0,SVM,basic_interpretable,1,1,1,0.331669,0.668331


In [15]:
metrics_df

,model,feature_set,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,roc_auc
0,KNN,basic_interpretable,0.634146,0.633201,0.634068,0.633201,0.633068,0.692372
1,RandomForest,basic_interpretable,0.663957,0.663360,0.663793,0.663360,0.663400,0.710288
2,SVM,basic_interpretable,0.691057,0.689286,0.693797,0.689286,0.688566,0.733451
3,XGBoost,basic_interpretable,0.642276,0.641931,0.642023,0.641931,0.641958,0.691093
